# Beijing PM2.5 Forecasting: LSTM vs GRU vs Encoder-Decoder

Each model uses the previous 24 hours to forecast PM2.5 for the next 24 hours (t+1 to t+24).
Results are reported in µg/m³ on the test set.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

I0000 00:00:1790766074.223408  950948 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790766074.295754  950948 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790766075.956615  950948 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Load data

In [5]:
HORIZON = 24

with open("artifacts/feature_config.json") as f:
    config = json.load(f)

with open("artifacts/target_scaler.json") as f:
    target_scaler = json.load(f)


def load_split(name):
    data = np.load(f"sequences/{name}_sequences.npz")
    return data["X"], data["y"][:, :HORIZON]


X_train, y_train = load_split("train")
X_val, y_val = load_split("validation")
X_test, y_test = load_split("test")

LOOKBACK = X_train.shape[1]
N_FEATURES = X_train.shape[2]

print("Train:     ", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:      ", X_test.shape, y_test.shape)

Train:      (247773, 24, 31) (247773, 24)
Validation: (46607, 24, 31) (46607, 24)
Test:       (50199, 24, 31) (50199, 24)


## Training setup

In [6]:
EPOCHS = 30
BATCH_SIZE = 256
UNITS = 64


def train(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2
        )
    ]

    return model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2
    )


models = {}
histories = {}

## 1. Vanilla LSTM

A single LSTM layer reads the input window. A dense layer then predicts all 24 horizons at once.

In [ ]:
def build_lstm():
    return keras.Sequential([
        layers.Input(shape=(LOOKBACK, N_FEATURES)),
        layers.LSTM(UNITS),
        layers.Dense(HORIZON)
    ], name="vanilla_lstm")


models["Vanilla LSTM"] = build_lstm()
models["Vanilla LSTM"].summary()

histories["Vanilla LSTM"] = train(models["Vanilla LSTM"])

Model: "vanilla_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        24,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,136 (102.09 KB)

 Trainable params: 26,136 (102.09 KB)

 Non-trainable params: 0 (0.00 B)

## 2. GRU

Same setup as the vanilla LSTM, with a GRU layer instead of the LSTM.

In [ ]:
def build_gru():
    return keras.Sequential([
        layers.Input(shape=(LOOKBACK, N_FEATURES)),
        layers.GRU(UNITS),
        layers.Dense(HORIZON)
    ], name="gru")


models["GRU"] = build_gru()
models["GRU"].summary()

histories["GRU"] = train(models["GRU"])

## 3. Encoder-Decoder LSTM (no attention)

The encoder compresses the input window into its final hidden and cell states.
The decoder starts from those states and produces one output per forecast step.

In [ ]:
def build_encoder_decoder():
    inputs = layers.Input(shape=(LOOKBACK, N_FEATURES))

    _, state_h, state_c = layers.LSTM(UNITS, return_state=True)(inputs)

    x = layers.RepeatVector(HORIZON)(state_h)
    x = layers.LSTM(UNITS, return_sequences=True)(x, initial_state=[state_h, state_c])
    x = layers.TimeDistributed(layers.Dense(1))(x)
    outputs = layers.Reshape((HORIZON,))(x)

    return keras.Model(inputs, outputs, name="encoder_decoder_lstm")


models["Encoder-Decoder LSTM"] = build_encoder_decoder()
models["Encoder-Decoder LSTM"].summary()

histories["Encoder-Decoder LSTM"] = train(models["Encoder-Decoder LSTM"])

## Training curves

In [ ]:
fig, axes = plt.subplots(1, len(histories), figsize=(18, 4), sharey=True)

for ax, (name, history) in zip(axes, histories.items()):
    ax.plot(history.history["loss"], label="train")
    ax.plot(history.history["val_loss"], label="validation")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE (scaled)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluation on the test set

Predictions are converted back to PM2.5 (µg/m³) before calculating RMSE, MAE and MAPE for each horizon.

In [ ]:
def inverse_scale(y):
    return y * target_scaler["std"] + target_scaler["mean"]


def horizon_metrics(y_true, y_pred):
    error = y_pred - y_true

    return pd.DataFrame({
        "horizon": np.arange(1, HORIZON + 1),
        "RMSE": np.sqrt(np.mean(error ** 2, axis=0)),
        "MAE": np.mean(np.abs(error), axis=0),
        "MAPE": np.mean(np.abs(error) / y_true, axis=0) * 100
    })


y_true = inverse_scale(y_test)

results = []

for name, model in models.items():
    y_pred = inverse_scale(model.predict(X_test, batch_size=1024, verbose=0))

    metrics = horizon_metrics(y_true, y_pred)
    metrics.insert(0, "model", name)
    results.append(metrics)

results = pd.concat(results, ignore_index=True)

In [ ]:
by_horizon = results.pivot(
    index="horizon",
    columns="model",
    values=["RMSE", "MAE", "MAPE"]
)
by_horizon.index = [f"t+{h}" for h in by_horizon.index]

display(by_horizon.round(2))

In [ ]:
summary = results.groupby("model")[["RMSE", "MAE", "MAPE"]].mean()

display(summary.round(2).sort_values("RMSE"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ["RMSE", "MAE", "MAPE"]):
    for name, group in results.groupby("model"):
        ax.plot(group["horizon"], group[metric], marker="o", label=name)

    ax.set_title(f"{metric} by Forecast Horizon")
    ax.set_xlabel("Horizon")
    ax.set_ylabel("MAPE (%)" if metric == "MAPE" else f"{metric} (µg/m³)")
    ax.set_xticks(range(1, HORIZON + 1))
    ax.set_xticklabels([f"t+{h}" for h in range(1, HORIZON + 1)], rotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## Save models and results

In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

for model in models.values():
    model.save(MODEL_DIR / f"{model.name}.keras")

results.to_csv("artifacts/test_metrics_by_horizon.csv", index=False)